# 🎯 Day 17 — Calibration, NAS, Normalising Flows & Fairness Constraints
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Probability Calibration | ~11 sec |
| 2 | 10:30–11:00 AM | Neural Architecture Search (NAS) | ~5 sec |
| 3 | 11:00 AM–1:00 PM | Normalising Flows + Fairness Constraints | ~20 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | Research Paper Pipeline Capstone | ~18 sec |

> **Zero downloads. Pure numpy + sklearn.**
---

In [ ]:
# !pip install scikit-learn matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize
import warnings, time, json
from datetime import datetime
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, load_wine, load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss
from sklearn.pipeline import make_pipeline
import sklearn
print('All imports OK')
print(f'   sklearn: {sklearn.__version__}')

---
## Session 1 — Probability Calibration
**Time:** 9:00–10:30 AM | **Run time: ~11 sec**

A model is **well-calibrated** when predicted probability 0.8 means the event occurs 80% of the time.
Most classifiers are overconfident — this session fixes that.

In [ ]:
# 1.1  Train an Overconfident Model
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.3, random_state=42)

gb = GradientBoostingClassifier(n_estimators=150, random_state=42).fit(X_tr, y_tr)
rf = RandomForestClassifier(150, random_state=42).fit(X_tr, y_tr)

prob_gb = gb.predict_proba(X_te)[:, 1]
prob_rf = rf.predict_proba(X_te)[:, 1]

def expected_calibration_error(y_true, y_prob, n_bins=10):
    """ECE: weighted average of |confidence - accuracy| across bins."""
    bins   = np.linspace(0, 1, n_bins + 1)
    ece    = 0.0
    n      = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0:
            continue
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += mask.sum() / n * abs(acc - conf)
    return ece

ece_gb = expected_calibration_error(y_te, prob_gb)
ece_rf = expected_calibration_error(y_te, prob_rf)
brier_gb = brier_score_loss(y_te, prob_gb)
brier_rf = brier_score_loss(y_te, prob_rf)

print('Uncalibrated Models:')
print(f'  GradientBoosting: ECE={ece_gb:.4f}  Brier={brier_gb:.4f}  AUC={roc_auc_score(y_te,prob_gb):.4f}')
print(f'  RandomForest    : ECE={ece_rf:.4f}  Brier={brier_rf:.4f}  AUC={roc_auc_score(y_te,prob_rf):.4f}')

In [ ]:
# 1.2  Reliability Diagram
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, prob, name, color in [(axes[0],prob_gb,'GradientBoosting','#534AB7'),
                               (axes[1],prob_rf,'RandomForest','#D85A30')]:
    frac_pos, mean_pred = calibration_curve(y_te, prob, n_bins=10)
    ax.plot([0,1],[0,1], 'k--', linewidth=1.5, alpha=0.5, label='Perfect calibration')
    ax.plot(mean_pred, frac_pos, 'o-', color=color, linewidth=2,
            markersize=6, label=f'{name}')
    ax.fill_between(mean_pred, frac_pos, mean_pred, alpha=0.1, color=color)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(f'{name}\nECE={expected_calibration_error(y_te,prob):.4f}')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    ax.set_xlim(0,1); ax.set_ylim(0,1)

plt.suptitle('Reliability Diagrams (closer to diagonal = better calibrated)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# 1.3  Platt Scaling (Logistic Regression on Scores)
def platt_scaling(scores_cal, y_cal, scores_test):
    """Fit logistic regression on held-out scores → calibrated probabilities."""
    lr_platt = LogisticRegression(max_iter=500)
    lr_platt.fit(scores_cal.reshape(-1,1), y_cal)
    return lr_platt.predict_proba(scores_test.reshape(-1,1))[:, 1]

# Use half of test set for calibration, half for evaluation
X_cal, X_eval, y_cal, y_eval = train_test_split(X_te, y_te, test_size=0.5, random_state=0)
prob_gb_cal  = gb.predict_proba(X_cal)[:, 1]
prob_gb_eval = gb.predict_proba(X_eval)[:, 1]

prob_platt = platt_scaling(prob_gb_cal, y_cal, prob_gb_eval)

ece_before = expected_calibration_error(y_eval, prob_gb_eval)
ece_platt  = expected_calibration_error(y_eval, prob_platt)
print(f'Platt Scaling:')
print(f'  ECE before : {ece_before:.4f}')
print(f'  ECE after  : {ece_platt:.4f}  (improvement: {(ece_before-ece_platt)/ece_before:.1%})')

In [ ]:
# 1.4  Temperature Scaling
def temperature_scale(logits, T):
    """Divide logits by T before softmax — only changes confidence, not predictions."""
    scaled = logits / T
    exp_s  = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_s / exp_s.sum(axis=1, keepdims=True)

def find_best_temperature(logits_cal, y_cal, T_range=np.linspace(0.1, 5.0, 100)):
    """Grid search for T minimising ECE on calibration set."""
    best_T, best_ece = 1.0, float('inf')
    for T in T_range:
        probs_T = temperature_scale(logits_cal, T)[:, 1]
        ece_T   = expected_calibration_error(y_cal, probs_T)
        if ece_T < best_ece:
            best_ece = ece_T; best_T = T
    return best_T, best_ece

# Approximate logits from GB probabilities: logit(p) = log(p/(1-p))
logits_cal  = np.column_stack([1-prob_gb_cal,  prob_gb_cal])
logits_eval = np.column_stack([1-prob_gb_eval, prob_gb_eval])
# Avoid log(0)
logits_cal  = np.log(np.clip(logits_cal,  1e-7, 1-1e-7))
logits_eval = np.log(np.clip(logits_eval, 1e-7, 1-1e-7))

best_T, best_ece_cal = find_best_temperature(logits_cal, y_cal)
prob_temp = temperature_scale(logits_eval, best_T)[:, 1]
ece_temp  = expected_calibration_error(y_eval, prob_temp)

print(f'Temperature Scaling:')
print(f'  Best T     : {best_T:.2f}  (T>1 = soften, T<1 = sharpen)')
print(f'  ECE before : {ece_before:.4f}')
print(f'  ECE after  : {ece_temp:.4f}')

# Compare all calibration methods
print('\nCalibration Method Comparison:')
for name, prob in [('Uncalibrated', prob_gb_eval),
                    ('Platt scaling', prob_platt),
                    ('Temperature',  prob_temp)]:
    ece_m   = expected_calibration_error(y_eval, prob)
    brier_m = brier_score_loss(y_eval, prob)
    auc_m   = roc_auc_score(y_eval, prob)
    print(f'  {name:20s}: ECE={ece_m:.4f}  Brier={brier_m:.4f}  AUC={auc_m:.4f}')

In [ ]:
# 1.5  Isotonic Regression Calibration + Visual Comparison
from sklearn.isotonic import IsotonicRegression

iso = IsotonicRegression(out_of_bounds='clip').fit(prob_gb_cal, y_cal)
prob_iso = iso.predict(prob_gb_eval)
ece_iso  = expected_calibration_error(y_eval, prob_iso)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, prob, name, color in [
    (axes[0], prob_gb_eval, 'Uncalibrated',    '#888780'),
    (axes[1], prob_platt,   'Platt scaling',   '#534AB7'),
    (axes[2], prob_iso,     'Isotonic regr.',  '#1D9E75'),
]:
    frac, mpred = calibration_curve(y_eval, prob, n_bins=8)
    ax.plot([0,1],[0,1],'k--',alpha=0.5)
    ax.plot(mpred, frac, 'o-', color=color, linewidth=2, markersize=6)
    ax.fill_between(mpred, frac, mpred, alpha=0.15, color=color)
    ece_v = expected_calibration_error(y_eval, prob)
    ax.set_title(f'{name}\nECE = {ece_v:.4f}')
    ax.set_xlabel('Mean predicted prob'); ax.set_ylabel('Fraction positive')
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.grid(alpha=0.3)

plt.suptitle('Calibration Methods: Reliability Diagrams', fontsize=12)
plt.tight_layout(); plt.show()

---
## Session 2 — Neural Architecture Search (NAS)
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  Random NAS: Search Over Architecture Hyperparameters
rng = np.random.default_rng(42)
X_nas, y_nas = load_wine(return_X_y=True)

# Search space: treat RF hyperparams as architecture choices
nas_results = []
for trial in range(25):
    n_trees   = int(rng.integers(10, 200))
    max_depth = int(rng.integers(2, 15))
    max_feats = float(rng.uniform(0.3, 1.0))
    min_split = int(rng.integers(2, 10))
    n_params  = n_trees * max_depth * int(max_feats * X_nas.shape[1])  # proxy for FLOPs

    model = RandomForestClassifier(
        n_estimators=n_trees, max_depth=max_depth,
        max_features=max_feats, min_samples_split=min_split,
        random_state=42
    )
    acc = cross_val_score(model, X_nas, y_nas, cv=3).mean()
    nas_results.append({'trial':trial+1, 'n_trees':n_trees, 'depth':max_depth,
                        'max_feats':round(max_feats,3), 'acc':round(acc,4),
                        'n_params':n_params})

df_nas = pd.DataFrame(nas_results).sort_values('acc', ascending=False)
print('NAS Top 5 Architectures:')
print(df_nas[['trial','n_trees','depth','max_feats','acc','n_params']].head(5).to_string(index=False))

In [ ]:
# 2.2  Pareto Frontier: Accuracy vs Model Size
def pareto_frontier(df, acc_col, size_col):
    """Find architectures on the Pareto frontier (best acc for given size)."""
    df_s     = df.sort_values(size_col)
    best_acc = -np.inf
    pareto   = []
    for _, row in df_s.iterrows():
        if row[acc_col] > best_acc:
            best_acc = row[acc_col]
            pareto.append(row)
    return pd.DataFrame(pareto)

pareto = pareto_frontier(df_nas, 'acc', 'n_params')

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df_nas['n_params'], df_nas['acc'], alpha=0.6, color='#888780',
           s=50, label='All architectures')
ax.plot(pareto['n_params'], pareto['acc'], 'o-', color='#534AB7',
        linewidth=2, markersize=8, label='Pareto frontier')
ax.set_xlabel('Proxy parameter count (model size)')
ax.set_ylabel('CV Accuracy')
ax.set_title('NAS Pareto Frontier: Accuracy vs Model Size')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'\nPareto-optimal architectures: {len(pareto)}')
print(f'Best accuracy: {pareto.acc.max():.4f}  '
      f'(n_trees={pareto.loc[pareto.acc.idxmax(),"n_trees"]}  '
      f'depth={pareto.loc[pareto.acc.idxmax(),"depth"]})')

In [ ]:
# 2.3  Progressive NAS: Start Small, Grow Architecture
print('Progressive NAS — growing architecture:')
prog_results = []
X_tr_nas, X_te_nas, y_tr_nas, y_te_nas = train_test_split(
    X_nas, y_nas, test_size=0.3, random_state=42
)

best_acc_prog = 0
best_config   = None

for n_trees in [10, 20, 50, 100, 150]:
    for max_depth in [3, 6, 10]:
        m = RandomForestClassifier(n_trees, max_depth=max_depth, random_state=42)
        m.fit(X_tr_nas, y_tr_nas)
        acc = m.score(X_te_nas, y_te_nas)
        prog_results.append({'n_trees':n_trees, 'depth':max_depth, 'acc':acc})
        if acc > best_acc_prog:
            best_acc_prog = acc; best_config = (n_trees, max_depth)

df_prog = pd.DataFrame(prog_results)
pivot = df_prog.pivot(index='depth', columns='n_trees', values='acc').round(4)
print('Accuracy heatmap (depth × n_trees):')
print(pivot)
print(f'\nProgressive NAS best: n_trees={best_config[0]}  depth={best_config[1]}  acc={best_acc_prog:.4f}')

---
## Session 3 — Normalising Flows + Fairness Constraints
**Time:** 11:00 AM–1:00 PM | **Run time: ~20 sec**

In [ ]:
# 3.1  Affine Coupling Layer (Core of RealNVP-style Flows)
class AffineCouplingLayer:
    """
    Split input: x = [x1, x2]
    Forward: y2 = x2 * exp(s(x1)) + t(x1)  — invertible!
    Inverse : x2 = (y2 - t(y1)) * exp(-s(y1))
    """
    def __init__(self, d_half, seed=42):
        rng_f = np.random.default_rng(seed)
        self.Ws = rng_f.normal(0, 0.1, (d_half, d_half))  # scale network weights
        self.Wt = rng_f.normal(0, 0.1, (d_half, d_half))  # translate network weights

    def _st(self, x1):
        s = np.tanh(x1 @ self.Ws)   # scale: bounded via tanh
        t = x1 @ self.Wt             # translate: unbounded
        return s, t

    def forward(self, x):
        """Data → latent. Returns (z, log_det_jacobian)."""
        d      = x.shape[1]
        x1, x2 = x[:, :d//2], x[:, d//2:]
        s, t   = self._st(x1)
        y2     = x2 * np.exp(s) + t
        log_det = s.sum(axis=1)      # log|det J| = sum of log scale factors
        return np.hstack([x1, y2]), log_det

    def inverse(self, z):
        """Latent → data."""
        d      = z.shape[1]
        z1, z2 = z[:, :d//2], z[:, d//2:]
        s, t   = self._st(z1)
        x2     = (z2 - t) * np.exp(-s)
        return np.hstack([z1, x2])

rng = np.random.default_rng(42)
X_flow = rng.normal(0, 1, (200, 8))
flow = AffineCouplingLayer(d_half=4, seed=42)

Z, log_det = flow.forward(X_flow)
X_rec = flow.inverse(Z)
max_err = np.abs(X_flow - X_rec).max()

print(f'Input shape       : {X_flow.shape}')
print(f'Latent shape      : {Z.shape}')
print(f'Log-det-Jacobian  : mean={log_det.mean():.4f}  std={log_det.std():.4f}')
print(f'Reconstruction err: {max_err:.2e}  (must be ~0 for valid flow)')

In [ ]:
# 3.2  Stack Multiple Coupling Layers (Deeper Flow)
class NormalisingFlow:
    """Stack of affine coupling layers with alternating splits."""
    def __init__(self, d, n_layers=4, seed=42):
        self.layers = [AffineCouplingLayer(d//2, seed=seed+i)
                       for i in range(n_layers)]
        self.d = d

    def forward(self, x):
        total_log_det = np.zeros(len(x))
        z = x.copy()
        for i, layer in enumerate(self.layers):
            if i % 2 == 1:  # alternate which half is transformed
                z = z[:, ::-1]
            z, ld = layer.forward(z)
            total_log_det += ld
            if i % 2 == 1:
                z = z[:, ::-1]
        return z, total_log_det

    def log_prob(self, x):
        """Log-likelihood under standard Normal prior."""
        z, log_det = self.forward(x)
        # log p(z) under standard normal
        log_pz = -0.5 * (z**2 + np.log(2*np.pi)).sum(axis=1)
        return log_pz + log_det

nflow = NormalisingFlow(d=8, n_layers=4)
log_probs = nflow.log_prob(X_flow)
print(f'\n4-layer normalising flow:')
print(f'  Log-prob mean : {log_probs.mean():.4f}')
print(f'  Log-prob std  : {log_probs.std():.4f}')

In [ ]:
# 3.3  Anomaly Detection via Flow Log-Likelihood
# Train on normal class, flag low log-prob as anomaly
rng = np.random.default_rng(7)
X_norm  = rng.normal(0, 1, (300, 8))     # normal distribution
X_anom  = rng.normal(4, 0.5, (30, 8))   # anomalous: shifted distribution
X_all   = np.vstack([X_norm, X_anom])
y_anom  = np.array([0]*300 + [1]*30)

# Fit flow on normal data
flow_anom = NormalisingFlow(d=8, n_layers=3)
lp_all    = flow_anom.log_prob(X_all)

# Threshold: flag samples below 5th percentile of normal log-probs
thresh = np.percentile(flow_anom.log_prob(X_norm), 5)
anomaly_pred = (lp_all < thresh).astype(int)

from sklearn.metrics import roc_auc_score
auc_flow = roc_auc_score(y_anom, -lp_all)  # lower log-prob = more anomalous
print(f'Flow anomaly detection AUC: {auc_flow:.4f}')
print(f'  Normal log-prob  : mean={flow_anom.log_prob(X_norm).mean():.2f}')
print(f'  Anomaly log-prob : mean={flow_anom.log_prob(X_anom).mean():.2f}')

In [ ]:
# 3.4  Fairness-Constrained Optimisation via Lagrangian Relaxation
rng = np.random.default_rng(42)
N   = 1500

# Synthetic loan dataset with gender bias
gender  = rng.choice([0,1], N, p=[0.55, 0.45])
credit  = rng.normal(650, 80, N)
income  = rng.exponential(45000, N)

# Features (credit and income only — no gender in feature set)
X_fair  = StandardScaler().fit_transform(
    np.column_stack([credit, income, rng.normal(0,1,(N,3))])
)
true_label = ((credit > 620) & (income > 30000)).astype(int)

# Step 1: Biased model (unconstrained)
X_tr_f, X_te_f, y_tr_f, y_te_f, g_tr_f, g_te_f = train_test_split(
    X_fair, true_label, gender, test_size=0.3, random_state=42
)
model_biased = LogisticRegression(max_iter=300).fit(X_tr_f, y_tr_f)
pred_biased  = model_biased.predict(X_te_f)

# Compute demographic parity gap
dp_gap_biased = abs(pred_biased[g_te_f==1].mean() - pred_biased[g_te_f==0].mean())
print(f'Biased model DP gap  : {dp_gap_biased:.4f}')

# Step 2: Constrained model — penalise demographic parity violation
def fairness_penalised_objective(coef, X_tr, y_tr, g_tr, lambda_fair=2.0):
    """Cross-entropy + λ * |P(ŷ=1|g=0) - P(ŷ=1|g=1)| fairness penalty."""
    logits = X_tr @ coef[:-1] + coef[-1]
    prob   = 1 / (1 + np.exp(-np.clip(logits, -30, 30)))
    # Cross-entropy loss
    ce     = -np.mean(y_tr*np.log(prob+1e-9) + (1-y_tr)*np.log(1-prob+1e-9))
    # Demographic parity penalty
    dp     = abs(prob[g_tr==0].mean() - prob[g_tr==1].mean())
    return ce + lambda_fair * dp

coef_init = np.zeros(X_tr_f.shape[1] + 1)
result    = optimize.minimize(
    fairness_penalised_objective, coef_init,
    args=(X_tr_f, y_tr_f, g_tr_f, 3.0),
    method='L-BFGS-B', options={'maxiter': 200}
)
coef_fair    = result.x
logits_fair  = X_te_f @ coef_fair[:-1] + coef_fair[-1]
pred_fair    = (logits_fair > 0).astype(int)
dp_gap_fair  = abs(pred_fair[g_te_f==1].mean() - pred_fair[g_te_f==0].mean())
acc_fair     = accuracy_score(y_te_f, pred_fair)
acc_biased   = accuracy_score(y_te_f, pred_biased)

print(f'Fair model DP gap    : {dp_gap_fair:.4f}  (improved by {dp_gap_biased-dp_gap_fair:.4f})')
print(f'Accuracy: biased={acc_biased:.4f}  fair={acc_fair:.4f}  (tradeoff: {acc_biased-acc_fair:.4f})')

---
## Lunch Break — 1:00–2:00 PM
1. Why does temperature scaling preserve accuracy but improve ECE?
2. What is the log-det-Jacobian and why is it needed in normalising flows?
3. How does Lagrangian relaxation convert a hard fairness constraint into a soft one?
4. What is the Pareto frontier in NAS multi-objective search?
---

## Session 5 — Research Paper Pipeline Capstone
**Time:** 2:00–4:00 PM | **Run time: ~18 sec**

In [ ]:
# 5.1  Full Research Pipeline: Calibrated + Fair + Anomaly-Aware Classifier
print('='*62)
print(' RESEARCH PAPER PIPELINE: CALIBRATED FAIR ML')
print('='*62)

# Dataset: Breast Cancer with synthetic protected attribute
X_r, y_r = load_breast_cancer(return_X_y=True)
rng_r    = np.random.default_rng(0)
group_r  = rng_r.choice([0,1], len(y_r), p=[0.55,0.45])  # synthetic protected attr

X_tr_r, X_tmp_r, y_tr_r, y_tmp_r, g_tr_r, g_tmp_r = train_test_split(
    X_r, y_r, group_r, test_size=0.4, random_state=42
)
X_cal_r, X_te_r, y_cal_r, y_te_r, g_cal_r, g_te_r = train_test_split(
    X_tmp_r, y_tmp_r, g_tmp_r, test_size=0.5, random_state=42
)

print(f'Train: {len(X_tr_r)}  Cal: {len(X_cal_r)}  Test: {len(X_te_r)}')

In [ ]:
# 5.2  Step 1: NAS-Found Architecture
# Use best config found in session 2 (simulated on breast cancer)
def nas_search(X_tr, y_tr, X_te, y_te, n_trials=15, seed=42):
    rng_n = np.random.default_rng(seed)
    best_acc, best_model = 0, None
    for _ in range(n_trials):
        n  = int(rng_n.integers(30, 150))
        d  = int(rng_n.integers(2, 10))
        f  = float(rng_n.uniform(0.4, 1.0))
        m  = GradientBoostingClassifier(
            n_estimators=n, max_depth=d, learning_rate=0.1, random_state=42
        ).fit(X_tr, y_tr)
        acc = m.score(X_te, y_te)
        if acc > best_acc:
            best_acc = acc; best_model = m
    return best_model, best_acc

best_model_r, nas_acc = nas_search(X_tr_r, y_tr_r, X_te_r, y_te_r, n_trials=15)
print(f'Step 1 — NAS best model test accuracy: {nas_acc:.4f}')

In [ ]:
# 5.3  Step 2: Calibrate with Isotonic Regression
from sklearn.isotonic import IsotonicRegression

prob_cal_r = best_model_r.predict_proba(X_cal_r)[:, 1]
iso_r      = IsotonicRegression(out_of_bounds='clip').fit(prob_cal_r, y_cal_r)

prob_test_raw  = best_model_r.predict_proba(X_te_r)[:, 1]
prob_test_cal  = iso_r.predict(prob_test_raw)

ece_raw = expected_calibration_error(y_te_r, prob_test_raw)
ece_cal = expected_calibration_error(y_te_r, prob_test_cal)
auc_r   = roc_auc_score(y_te_r, prob_test_cal)

print(f'Step 2 — Calibration:')
print(f'  ECE before : {ece_raw:.4f}')
print(f'  ECE after  : {ece_cal:.4f}  (improvement: {(ece_raw-ece_cal)/ece_raw:.1%})')
print(f'  AUC        : {auc_r:.4f}')

In [ ]:
# 5.4  Step 3: Fairness Audit + Report
pred_r = (prob_test_cal >= 0.5).astype(int)

def fairness_report(y_true, y_pred, group, names):
    rows = []
    for g, name in zip(np.unique(group), names):
        m  = group == g
        tp = ((y_pred[m]==1)&(y_true[m]==1)).sum()
        fn = ((y_pred[m]==0)&(y_true[m]==1)).sum()
        fp = ((y_pred[m]==1)&(y_true[m]==0)).sum()
        tn = ((y_pred[m]==0)&(y_true[m]==0)).sum()
        rows.append({'Group':name,
                     'TPR':round(tp/(tp+fn+1e-9),4),
                     'FPR':round(fp/(fp+tn+1e-9),4),
                     'PosRate':round(y_pred[m].mean(),4)})
    df_f = pd.DataFrame(rows)
    df_f['DI_ratio'] = (df_f['PosRate']/df_f['PosRate'].max()).round(4)
    return df_f

fair_df = fairness_report(y_te_r, pred_r, g_te_r, ['Group 0','Group 1'])
print('Step 3 — Fairness Audit:')
print(fair_df.to_string(index=False))
di = fair_df['DI_ratio'].min()
print(f'\nDisparate Impact ratio: {di:.4f}  (>0.8 = fair: {di>=0.8})')

In [ ]:
# 5.5  Step 4: Flow-Based Anomaly Flagging on Test Set
scaler_r = StandardScaler().fit(X_tr_r)
X_tr_sc_r = scaler_r.transform(X_tr_r)
X_te_sc_r = scaler_r.transform(X_te_r)

# Fit flow on training data
flow_r = NormalisingFlow(d=X_tr_sc_r.shape[1] if X_tr_sc_r.shape[1]%2==0
                          else X_tr_sc_r.shape[1]-1,
                          n_layers=3)
# Truncate to even number of features for flow
d_even = (X_tr_sc_r.shape[1]//2)*2
X_tr_even = X_tr_sc_r[:, :d_even]
X_te_even = X_te_sc_r[:, :d_even]
flow_r    = NormalisingFlow(d=d_even, n_layers=3)
lp_train  = flow_r.log_prob(X_tr_even)
lp_test   = flow_r.log_prob(X_te_even)
anom_thr  = np.percentile(lp_train, 5)
anom_flag = (lp_test < anom_thr).astype(int)
print(f'Step 4 — Flow anomaly detection:')
print(f'  Flagged {anom_flag.sum()} / {len(anom_flag)} test samples as anomalous ({anom_flag.mean():.1%})')

In [ ]:
# 5.6  Final Research Report + Visualisation
print('\n' + '='*62)
print(' FINAL RESEARCH REPORT')
print('='*62)
print(f' Date           : {datetime.utcnow().strftime("%Y-%m-%d")}')
print(f' Title          : Calibrated Fair ML with Flow Anomaly Detection')
print()
print(' COMPONENTS:')
print(f'   NAS-found architecture accuracy   : {nas_acc:.4f}')
print(f'   ECE before calibration            : {ece_raw:.4f}')
print(f'   ECE after isotonic calibration    : {ece_cal:.4f}')
print(f'   AUC-ROC                           : {auc_r:.4f}')
print(f'   Disparate impact ratio            : {di:.4f}  (fair={di>=0.8})')
print(f'   Anomaly flagging rate             : {anom_flag.mean():.1%}')
print()
print(' SKILLS DEMONSTRATED:')
for s in ['NAS search space exploration + Pareto frontier',
          'Isotonic regression probability calibration',
          'ECE measurement before and after calibration',
          'Fairness audit: TPR parity + disparate impact ratio',
          'Normalising flow log-likelihood anomaly detection',
          'Lagrangian fairness constraint optimisation']:
    print(f'   + {s}')
print('='*62)

# Visualise reliability diagram of final model
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Reliability diagram
frac_v, mp_v = calibration_curve(y_te_r, prob_test_cal, n_bins=8)
axes[0].plot([0,1],[0,1],'k--',alpha=0.5)
axes[0].plot(mp_v, frac_v, 'o-', color='#534AB7', linewidth=2, markersize=7)
axes[0].fill_between(mp_v, frac_v, mp_v, alpha=0.15, color='#534AB7')
axes[0].set_title(f'Reliability Diagram\nECE={ece_cal:.4f}')
axes[0].set_xlabel('Mean predicted prob'); axes[0].set_ylabel('Fraction positive')
axes[0].grid(alpha=0.3)

# Fairness: positive rate per group
axes[1].bar(fair_df['Group'], fair_df['PosRate'], color=['#534AB7','#1D9E75'], alpha=0.85)
axes[1].axhline(fair_df['PosRate'].max()*0.8, linestyle='--', color='#D85A30',
                label='80% DI threshold')
axes[1].set_title(f'Demographic Parity\nDI={di:.4f}')
axes[1].set_ylabel('Positive rate'); axes[1].legend(fontsize=8)

# Flow log-prob distribution
axes[2].hist(lp_train, bins=20, alpha=0.6, color='#534AB7', label='Train', density=True)
axes[2].hist(lp_test[anom_flag==0], bins=15, alpha=0.6, color='#1D9E75', label='Normal test', density=True)
axes[2].hist(lp_test[anom_flag==1], bins=10, alpha=0.8, color='#D85A30', label='Anomalous', density=True)
axes[2].axvline(anom_thr, linestyle='--', color='black', linewidth=1.5, label='Threshold')
axes[2].set_title('Flow Log-Prob Distribution')
axes[2].set_xlabel('Log-likelihood'); axes[2].legend(fontsize=7)

plt.suptitle('Research Pipeline: Calibrated + Fair + Anomaly-Aware ML', fontsize=12)
plt.tight_layout(); plt.show()

---
## Day 17 Summary

| Concept | Skill Gained |
|---|---|
| Reliability diagram | Bin predictions, plot fraction vs confidence |
| ECE | Weighted average miscalibration across bins |
| Platt scaling | Logistic regression post-hoc calibration on scores |
| Temperature scaling | Grid search T to minimise ECE on calibration set |
| Isotonic regression | Non-parametric monotone calibration |
| Random NAS | Sample architecture hyperparams, evaluate, rank |
| Pareto frontier | Accuracy vs size trade-off for architecture selection |
| Progressive NAS | Grow architecture iteratively, keep best |
| Affine coupling layer | Split, scale+translate, exactly invertible |
| Normalising flow stack | Multiple coupling layers, alternating splits |
| Flow anomaly detection | Low log-likelihood = anomalous sample |
| Fairness constraint | Lagrangian relaxation with demographic parity penalty |
| Research pipeline | NAS → calibrate → fairness audit → anomaly detection |

---
## Day 18 Preview
- Reinforcement learning from human feedback (RLHF) concept
- Reward modelling: learning preferences from pairwise comparisons
- PPO (Proximal Policy Optimisation) in tabular RL
- LLM fine-tuning patterns and prompt tuning
- Capstone: build an RLHF-style preference model

In [ ]:
# Bonus: Day 17 in one cell
X_b, y_b = load_breast_cancer(return_X_y=True)
Xtr_b,Xte_b,ytr_b,yte_b = train_test_split(X_b,y_b,test_size=0.3,random_state=42)
gb_b = GradientBoostingClassifier(100,random_state=42).fit(Xtr_b,ytr_b)
prob_b = gb_b.predict_proba(Xte_b)[:,1]
ece_b  = expected_calibration_error(yte_b, prob_b)
print(f'ECE before calibration: {ece_b:.4f}')

Xcal_b,Xev_b,ycal_b,yev_b = train_test_split(Xte_b,yte_b,test_size=0.5,random_state=0)
iso_b = IsotonicRegression(out_of_bounds='clip').fit(gb_b.predict_proba(Xcal_b)[:,1], ycal_b)
prob_b2 = iso_b.predict(gb_b.predict_proba(Xev_b)[:,1])
print(f'ECE after  calibration: {expected_calibration_error(yev_b,prob_b2):.4f}')

flow_b = NormalisingFlow(d=8, n_layers=2)
rng_b2 = np.random.default_rng(42)
X_b3 = rng_b2.normal(0,1,(50,8))
lp_b = flow_b.log_prob(X_b3)
print(f'Flow log-prob mean: {lp_b.mean():.4f}')
print('\nDay 17 complete — calibration, NAS, normalising flows, fairness constraints!')